In [199]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit,cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, 
                             f1_score, roc_auc_score, roc_curve, auc, make_scorer)

from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE



## 1. Load Data & Merged Data

In [200]:
songs_labeled = pd.read_csv('songs_aggregated_labeled.csv', sep=';')

songs_labeled['first_appearance'] = pd.to_datetime(songs_labeled['first_appearance'])
songs_labeled['last_appearance'] = pd.to_datetime(songs_labeled['last_appearance'])

print(f"  Total songs: {len(songs_labeled):,}")
print(f"  Training:    {songs_labeled['is_train'].sum():,}")
print(f"  Test:        {(~songs_labeled['is_train']).sum():,}")

songs_labeled.head()

  Total songs: 9,161
  Training:    7,572
  Test:        1,589


,id,best_rank,avg_rank,total_weeks_charted,first_appearance,last_appearance,Title,Artist,peak_score,longevity_score,popularity_score,popularity_label,is_train
0,000xQL6tZNLJzIrtIgxqSl,40,101.500000,118,2017-03-24,2017-09-12,Still Got Time (feat. PARTYNEXTDOOR),ZAYN,80.5,100.0,92.2,Popular,True
1,003VDDA7J3Xb2ZFlNx7nIZ,108,138.000000,2,2020-02-07,2020-02-08,YELL OH,Trippie Redd,46.5,10.0,24.6,Not Popular,True
2,003eoIwxETJujVWmNFMoZy,91,136.500000,14,2018-06-15,2018-06-28,Growing Pains,Alessia Cara,55.0,70.0,64.0,Not Popular,True
3,003vvx7Niy0yvhvHt4a68B,73,173.794872,273,2020-08-15,2021-12-31,Mr. Brightside,The Killers,64.0,100.0,85.6,Popular,True
4,00B7TZ0Xawar6NZ00JFomN,61,109.285714,14,2018-04-06,2018-04-19,Best Life (feat. Chance The Rapper),Cardi B,70.0,70.0,70.0,Not Popular,True


In [201]:
df_song_charts = pd.read_csv("final_cleaned_data.csv", sep=';')
df_song_charts['Date'] = pd.to_datetime(df_song_charts['Date'])
audio_features = df_song_charts.groupby('id')[['Danceability', 'Energy', 'Loudness_norm', 
                                                    'Speechiness', 'Acousticness', 
                                                    'Instrumentalness',  'Valence'
                                                    ]].first().reset_index()

In [202]:
merged_df_spotify = songs_labeled.merge(audio_features, on='id', how='inner')
merged_df_spotify.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9161 entries, 0 to 9160
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   9161 non-null   object        
 1   best_rank            9161 non-null   int64         
 2   avg_rank             9161 non-null   float64       
 3   total_weeks_charted  9161 non-null   int64         
 4   first_appearance     9161 non-null   datetime64[ns]
 5   last_appearance      9161 non-null   datetime64[ns]
 6   Title                9161 non-null   object        
 7   Artist               9161 non-null   object        
 8   peak_score           9161 non-null   float64       
 9   longevity_score      9161 non-null   float64       
 10  popularity_score     9161 non-null   float64       
 11  popularity_label     9161 non-null   object        
 12  is_train             9161 non-null   bool          
 13  Danceability         9161 non-nul

In [203]:
merged_df_spotify['release_year'] = merged_df_spotify['first_appearance'].dt.year
merged_df_spotify['release_month'] = merged_df_spotify['first_appearance'].dt.month
merged_df_spotify['release_quarter'] = merged_df_spotify['first_appearance'].dt.quarter
merged_df_spotify['release_day_of_week'] = merged_df_spotify['first_appearance'].dt.dayofweek
merged_df_spotify['chart_lifespan_days'] = (merged_df_spotify['last_appearance'] - merged_df_spotify['first_appearance']).dt.days

merged_df_spotify['is_holiday_season'] = (
   merged_df_spotify['release_month'].isin([11, 12])
).astype(int)

merged_df_spotify['is_friday_release'] = (
   merged_df_spotify['release_day_of_week'] == 4
).astype(int)
merged_df_spotify.head()


,id,best_rank,avg_rank,total_weeks_charted,first_appearance,last_appearance,Title,Artist,peak_score,longevity_score,...,Acousticness,Instrumentalness,Valence,release_year,release_month,release_quarter,release_day_of_week,chart_lifespan_days,is_holiday_season,is_friday_release
0,000xQL6tZNLJzIrtIgxqSl,40,101.500000,118,2017-03-24,2017-09-12,Still Got Time (feat. PARTYNEXTDOOR),ZAYN,80.5,100.0,...,0.131,0.0,0.524,2017,3,1,4,172,0,1
1,003VDDA7J3Xb2ZFlNx7nIZ,108,138.000000,2,2020-02-07,2020-02-08,YELL OH,Trippie Redd,46.5,10.0,...,0.004,0.0,0.190,2020,2,1,4,1,0,1
2,003eoIwxETJujVWmNFMoZy,91,136.500000,14,2018-06-15,2018-06-28,Growing Pains,Alessia Cara,55.0,70.0,...,0.082,0.0,0.437,2018,6,2,4,13,0,1
3,003vvx7Niy0yvhvHt4a68B,73,173.794872,273,2020-08-15,2021-12-31,Mr. Brightside,The Killers,64.0,100.0,...,0.001,0.0,0.236,2020,8,3,5,503,0,0
4,00B7TZ0Xawar6NZ00JFomN,61,109.285714,14,2018-04-06,2018-04-19,Best Life (feat. Chance The Rapper),Cardi B,70.0,70.0,...,0.287,0.0,0.665,2018,4,2,4,13,0,1


In [204]:
audio_features_cols = ['Danceability', 'Energy', 'Loudness_norm', 'Speechiness', 
                       'Acousticness', 'Instrumentalness', 'Valence']

artist_features_cols = [ 
                        'artist_song_count', 
                        'artist_total_appearances']

temporal_features_cols = ['release_month', 'release_quarter', 
                          'release_day_of_week', 'chart_lifespan_days', 'is_holiday_season', 'is_friday_release']

all_features = audio_features_cols + artist_features_cols + temporal_features_cols


In [205]:
artist_stats_train = df_song_charts.groupby('Artist (Ind.)').agg({
    'id': 'nunique',           # Number of unique songs
    'Rank': 'min',              # Best rank achieved
    'Date': 'count'           # Total appearances
}).reset_index()

artist_stats_train.columns = ['Artist', 'artist_song_count', 'artist_best_rank', 'artist_total_appearances']
merged_df = merged_df_spotify.merge(artist_stats_train, on='Artist', how='left')


## 2. Model validation
We chose 2 different model validation method which are Train Test Split and TimeSeriesSplitter. The reason that we need to use TimeSeriesSplitter is because we don't want our model to accidentally train on future data

In [206]:
merged_df['Year'] = merged_df['first_appearance'].dt.year
train_years = [2017, 2018, 2019, 2020, 2021]
test_years = [2022, 2023]

training_merged_df = merged_df[merged_df['Year'].isin(train_years)].copy()
test_merged_df = merged_df[merged_df['Year'].isin(test_years)].copy()


X_train = training_merged_df[all_features].copy()
y_train = (training_merged_df['popularity_label'] == 'Popular').astype(int)
X_test = test_merged_df[all_features].copy()
y_test = (test_merged_df['popularity_label'] == 'Popular').astype(int)


In [207]:
tscv = TimeSeriesSplit(n_splits=5)

## 3. Scaling

For Train Test Splitting + TimeSeriesSPlitter

In [208]:
def create_preprocessor(model_type, features):
    if model_type in ['RandomForest', 'XGBoost', 'DecisionTree']:
        return ColumnTransformer(
            [('passthrough', 'passthrough', features)]
        )
    
    elif model_type in ['KNN']:
        return ColumnTransformer([
            ('minmaxscaler', MinMaxScaler(), features) 
        ])
    else:
        return ColumnTransformer(
            [('scaler', StandardScaler(), features)]
        )

In [209]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), all_features)
    ])

## 4. Sampling

Train Test Split + Time SeriesSplitter

In [210]:
smote = SMOTE(random_state=42)


## 5. Train Model  

In [211]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'SVM': SVC(random_state=42, probability=True),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB()
}

Train Test Split

In [212]:
train_test_split_results = []
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', create_preprocessor(name,all_features)),
        ('sampler', SMOTE(random_state=42)),
        ('classifier', model)
    ])


    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    result = {}
    result['Model Name'] = name
    if hasattr(model, 'predict_proba'):
        y_pred_proba = pipeline.predict_proba(X_test)[:, 1] # Probability of the positive class (1)
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        result['ROC_AUC'] = roc_auc
    else:
        result['ROC_AUC'] = 'Not Available'
    result['Test_Score'] = pipeline.score(X_test,y_test)
    result['F1-Score'] = f1 = f1_score(y_test, y_pred)
    train_test_split_results.append(result)
    



train_test_df = pd.DataFrame(train_test_split_results)

In [214]:
sorted_merged_df = merged_df.sort_values('first_appearance')
sorted_X = sorted_merged_df[all_features].copy()
sorted_y = (sorted_merged_df['popularity_label'] == 'Popular').astype(int)

cv_results = []
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', create_preprocessor(name,all_features)),
        ('sampler', SMOTE(random_state=42)),
        ('classifier', model)
    ])
    scores = cross_val_score(pipeline, sorted_X, sorted_y, cv=tscv, scoring='roc_auc')
    cv_results.append({'Model Name': name, 'CV_Mean': scores.mean(), 'CV_Std': scores.std()})
    
cv_df = pd.DataFrame(cv_results)

comparison_df = train_test_df.merge(cv_df, on='Model Name', how='outer')
print(comparison_df)

            Model Name   ROC_AUC  Test_Score  F1-Score   CV_Mean    CV_Std
0                  KNN  0.618157    0.614223  0.477408  0.632629  0.022767
1  Logistic Regression  0.814651    0.774701  0.609170  0.792129  0.030881
2          Naive Bayes  0.647902    0.616740  0.506883  0.675818  0.026453
3        Random Forest  0.968650    0.904342  0.850394  0.960156  0.012984
4                  SVM  0.908021    0.831340  0.709957  0.883298  0.025830
5              XGBoost  0.967240    0.893014  0.825103  0.960614  0.010221
